In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
base_path = "/content/drive/MyDrive/GDA_Project/"

In [ ]:
import pandas as pd

base_path = "/content/drive/MyDrive/GDA_Project/"

normal = pd.read_csv(base_path + "BC-TCGA-Normal.txt", sep="\t", index_col=0)
tumor = pd.read_csv(base_path + "BC-TCGA-Tumor.txt", sep="\t", index_col=0)

print("Normal:", normal.shape)
print("Tumor:", tumor.shape)

Normal: (17814, 61)
Tumor: (17814, 529)


In [ ]:
#combine data
expression = pd.concat([normal, tumor], axis=1)

print("Combined:", expression.shape)

Combined: (17814, 590)


In [ ]:
print(expression.index[:10])  # genes
print(expression.columns[:5]) # samples

Index(['ELMO2', 'CREB3L1', 'RPS11', 'PNMA1', 'MMP2', 'C10orf90', 'ZHX3',
       'ERCC5', 'GPR98', 'RXFP3'],
      dtype='object', name='Hybridization REF')
Index(['TCGA-BH-A0AY-11A-23R-A089-07', 'TCGA-A7-A0DB-11A-33R-A089-07',
       'TCGA-BH-A0HK-11A-11R-A089-07', 'TCGA-BH-A0BM-11A-12R-A089-07',
       'TCGA-BH-A0B3-11B-21R-A089-07'],
      dtype='object')


In [ ]:
#creating tf and sf lists manually for baseline correlation
tf_list = [
    "TP53", "MYC", "FOXA1", "GATA3", "STAT3",
    "NFKB1", "ESR1", "JUN", "FOS", "SP1"
]

In [ ]:
sf_list = [
    "SRSF1", "HNRNPA1", "RBFOX2", "PTBP1",
    "U2AF2", "SRSF3", "SRSF7"
]

In [ ]:
#Matching genes
tf_genes = [g for g in tf_list if g in expression.index]
sf_genes = [g for g in sf_list if g in expression.index]

print("TF found:", tf_genes)
print("SF found:", sf_genes)

TF found: ['TP53', 'MYC', 'FOXA1', 'GATA3', 'STAT3', 'NFKB1', 'ESR1', 'JUN', 'FOS', 'SP1']
SF found: ['HNRNPA1', 'PTBP1', 'U2AF2']


In [ ]:
#Extract expression
tf_expr = expression.loc[tf_genes]
sf_expr = expression.loc[sf_genes]

In [ ]:
#Simple network based on correlation
import numpy as np

edges = []

for tf in tf_genes:
    for gene in expression.index:
        corr = np.corrcoef(expression.loc[tf], expression.loc[gene])[0,1]
        if abs(corr) > 0.7:
            edges.append((tf, gene, corr))

print("Edges found:", len(edges))

Edges found: 121


In [ ]:
#Creating a graph
import networkx as nx

G = nx.DiGraph()

for src, tgt, w in edges:
    G.add_edge(src, tgt, weight=w)

print("Nodes:", G.number_of_nodes())
print("Edges:", G.number_of_edges())

Nodes: 77
Edges: 121


In [ ]:
#Compute differential expression
# mean expression
normal_mean = normal.mean(axis=1)
tumor_mean = tumor.mean(axis=1)

# difference
diff = abs(tumor_mean - normal_mean)

In [ ]:
#select top differential genes
top_diff_genes = diff.sort_values(ascending=False).head(2000).index

In [ ]:
#Loading and cleaning TF list
tf_raw = pd.read_csv(base_path + "Homo_sapiens_TF.txt", sep="\t")

In [ ]:
print(tf_raw.columns)

Index(['Species', 'Symbol', 'Ensembl', 'Family', 'Protein', 'Entrez_ID'], dtype='object')


In [ ]:
tf_list = tf_raw["Symbol"].dropna().unique().tolist()

In [ ]:
tf_genes = [g for g in tf_list if g in expression.index]

print("Total TFs in database:", len(tf_list))
print("TFs found in your dataset:", len(tf_genes))

Total TFs in database: 1637
TFs found in your dataset: 1367


In [ ]:
selected_genes = list(set(top_diff_genes) | set(tf_genes))